1. Run the Apriori Notebook Shared by me on basket dataset using different Support and confidence values.


In [12]:
from collections import Counter,OrderedDict

In [9]:
import pyECLAT
import pandas as pd
from itertools import combinations

data = pd.read_csv(r"D:\College_work\ML\LAB_6\basket.csv")
print(data.head())
data = data.fillna('')
data.columns = range(len(data.columns))
print("Shape:", data.shape)
print(data.head())

   LBE Brooklyn             11204       Unnamed: 3 Unnamed: 4  Unnamed: 5
0  MBE      WBE             BLACK  Cambria Heights      11411         NaN
1  MBE    BLACK  Yorktown Heights            10598        NaN         NaN
2  MBE    BLACK        Long Beach            11561        NaN         NaN
3  MBE    ASIAN          Brooklyn            11235        NaN         NaN
4  MBE      WBE             ASIAN         New York      10010         NaN
Shape: (1419, 6)
     0      1                 2                3      4 5
0  MBE    WBE             BLACK  Cambria Heights  11411  
1  MBE  BLACK  Yorktown Heights            10598         
2  MBE  BLACK        Long Beach            11561         
3  MBE  ASIAN          Brooklyn            11235         
4  MBE    WBE             ASIAN         New York  10010  


In [11]:
VALID_CATEGORIES = {'MBE','WBE','LBE','BLACK','ASIAN','HISPANIC',
                    'NON-MINORITY','NATIVE AMERICAN','EBE','N/A'}

def clean_row(row):
    return [str(v).strip() for v in row if str(v).strip() in VALID_CATEGORIES]

transactions = [clean_row(data.iloc[i]) for i in range(len(data))]
transactions = [t for t in transactions if len(t) >= 2]  # drop rows with < 2 items

print("Total valid transactions:", len(transactions))
print("Sample:", transactions)

Total valid transactions: 1379
Sample: [['MBE', 'WBE', 'BLACK'], ['MBE', 'BLACK'], ['MBE', 'BLACK'], ['MBE', 'ASIAN'], ['MBE', 'WBE', 'ASIAN'], ['MBE', 'ASIAN'], ['MBE', 'BLACK'], ['MBE', 'HISPANIC'], ['MBE', 'WBE', 'BLACK'], ['MBE', 'ASIAN'], ['MBE', 'WBE', 'HISPANIC'], ['MBE', 'WBE', 'ASIAN'], ['MBE', 'BLACK'], ['WBE', 'NON-MINORITY'], ['MBE', 'BLACK'], ['MBE', 'BLACK'], ['WBE', 'NON-MINORITY'], ['WBE', 'NON-MINORITY'], ['WBE', 'NON-MINORITY'], ['MBE', 'BLACK'], ['MBE', 'HISPANIC'], ['MBE', 'WBE', 'BLACK'], ['WBE', 'NON-MINORITY'], ['MBE', 'ASIAN'], ['WBE', 'NON-MINORITY'], ['MBE', 'WBE', 'BLACK'], ['MBE', 'HISPANIC'], ['MBE', 'ASIAN'], ['MBE', 'BLACK'], ['WBE', 'NON-MINORITY'], ['MBE', 'LBE', 'ASIAN'], ['MBE', 'HISPANIC'], ['MBE', 'BLACK'], ['WBE', 'NON-MINORITY'], ['MBE', 'ASIAN'], ['MBE', 'WBE', 'LBE', 'HISPANIC'], ['MBE', 'ASIAN'], ['MBE', 'WBE', 'HISPANIC'], ['MBE', 'LBE', 'BLACK'], ['MBE', 'WBE', 'BLACK'], ['MBE', 'ASIAN'], ['MBE', 'BLACK'], ['MBE', 'ASIAN'], ['MBE', 'ASIAN'], 

In [12]:
max_len  = max(len(t) for t in transactions)
clean_df = pd.DataFrame([t + [''] * (max_len - len(t)) for t in transactions])
clean_df.columns = range(len(clean_df.columns))
print(clean_df.head())

     0      1      2 3
0  MBE    WBE  BLACK  
1  MBE  BLACK         
2  MBE  BLACK         
3  MBE  ASIAN         
4  MBE    WBE  ASIAN  


In [13]:
def get_support(itemset, transactions):
    count = sum(1 for t in transactions if all(i in t for i in itemset))
    return count / len(transactions)

def generate_rules(itemset, transactions, min_confidence):
    rules = []
    items = list(itemset)
    for r in range(1, len(items)):
        for antecedent in combinations(items, r):
            consequent = tuple(i for i in items if i not in antecedent)
            if not consequent:
                continue
            sup_ant  = get_support(list(antecedent), transactions)
            sup_full = get_support(items, transactions)
            if sup_ant == 0:
                continue
            confidence = sup_full / sup_ant
            if confidence >= min_confidence:
                lift = confidence / get_support(list(consequent), transactions)
                rules.append({
                    'antecedents': set(antecedent),
                    'consequents': set(consequent),
                    'support'    : round(sup_full, 3),
                    'confidence' : round(confidence, 3),
                    'lift'       : round(lift, 3)
                })
    return rules

print("Helper functions defined.")

Helper functions defined.


In [15]:
support_values    = [0.05, 0.1, 0.3, 0.5]
confidence_values = [0.5, 0.6, 0.7, 0.8]

for sup in support_values:
    eclat = ECLAT(data=clean_df, verbose=False)
    _, rule_supports = eclat.fit(min_support=sup, min_combination=2,
                                  max_combination=len(clean_df.columns))
    frequent_itemsets = [key.split(' & ') for key in rule_supports.keys()]

    for conf in confidence_values:
        all_rules = []
        for itemset in frequent_itemsets:
            all_rules.extend(generate_rules(itemset, transactions, conf))

        print(f"\n===== Support={sup}  Confidence={conf} → {len(all_rules)} rules =====")
        if all_rules:
            print(pd.DataFrame(all_rules).to_string(index=False))
        else:
            print("  (no rules generated)")

Combination 2 by 2


0it [00:00, ?it/s]

21it [00:00, 260.01it/s]


Combination 3 by 3


35it [00:00, 289.23it/s]


...

Combination 3 by 3


0it [00:00, ?it/s]


Combination 4 by 4


0it [00:00, ?it/s]



===== Support=0.5  Confidence=0.5 → 0 rules =====
  (no rules generated)

===== Support=0.5  Confidence=0.6 → 0 rules =====
  (no rules generated)

===== Support=0.5  Confidence=0.7 → 0 rules =====
  (no rules generated)

===== Support=0.5  Confidence=0.8 → 0 rules =====
  (no rules generated)


2. What is maximum size of rule that can be created?

In [16]:
MIN_SUPPORT    = 0.05
MIN_CONFIDENCE = 0.5

eclat = ECLAT(data=clean_df, verbose=False)
_, rule_supports = eclat.fit(min_support=MIN_SUPPORT, min_combination=2,
                              max_combination=len(clean_df.columns))
frequent_itemsets = [key.split(' & ') for key in rule_supports.keys()]

print("Frequent itemsets found:", len(frequent_itemsets))

Combination 2 by 2


21it [00:00, 266.17it/s]


Combination 3 by 3


35it [00:00, 327.99it/s]


Combination 4 by 4


35it [00:00, 332.90it/s]


Frequent itemsets found: 20



3. At what Confidence value, Minimum number of rules are generated.

In [17]:
max_size        = 0
max_rule_detail = None

for itemset in frequent_itemsets:
    for rule in generate_rules(itemset, transactions, MIN_CONFIDENCE):
        size = len(rule['antecedents']) + len(rule['consequents'])
        if size > max_size:
            max_size        = size
            max_rule_detail = rule

print(f"Maximum rule size : {max_size} items")
print(f"Antecedents       : {max_rule_detail['antecedents']}")
print(f"Consequents       : {max_rule_detail['consequents']}")
print(f"Support           : {max_rule_detail['support']}")
print(f"Confidence        : {max_rule_detail['confidence']}")
print(f"Lift              : {max_rule_detail['lift']}")

Maximum rule size : 3 items
Antecedents       : {'BLACK', 'WBE'}
Consequents       : {'MBE'}
Support           : 0.084
Confidence        : 1.0
Lift              : 1.45
